In [2]:
import os
import numpy as np
import pandas as pd

Now im going to run ADF test on these stocks to see if they are stationary or not

In [ ]:
selected_tickers = [ "GARAN.IS", "AKBNK.IS", "ISCTR.IS", "YKBNK.IS", "TUPRS.IS", 
    "EREGL.IS", "KCHOL.IS", "SAHOL.IS", "SISE.IS", "THYAO.IS", "BIMAS.IS" ]

print("--- Results ---")

for ticker_name in selected_tickers:
    ticker_name = ticker_name.replace(".IS", "")
    df = pd.read_csv(f"../data/{ticker_name}.csv")

    df['daily_change'] = df['Close'].diff()
    df['lag'] = df['Close'].shift(1)

    df_clean = df.dropna(subset=['daily_change', 'lag'])

    ones_col = np.ones(len(df_clean))
    matrix = np.column_stack((ones_col, df_clean['lag'].values))
    matrix_t = matrix.T

    matrix_multiplication = matrix_t @ matrix
    inv_matrix_multiplication = np.linalg.inv(matrix_multiplication)

    y = df_clean['daily_change'].values
    coefficients = inv_matrix_multiplication @ (matrix_t  @ y)

    intercept = coefficients[0] #Drift
    gamma = coefficients[1] #Gamma in ADF test

    y_pred = matrix @ coefficients #Predicting data according to our model
    residuals = y - y_pred  #Finding the error between the predicted data nad our real data

    n = len(y)
    degrees_of_freedom = n - 2
    residual_variance = np.sum(residuals ** 2) / degrees_of_freedom

    vcov = residual_variance * inv_matrix_multiplication

    se_gamma = np.sqrt(vcov[1,1])

    adf_stat = coefficients[1] / se_gamma #This value is our Dickey-Fuller test statistic

    print(f"Results for {ticker_name}: ")
    print(f"Gamma (γ): {coefficients[1]:.6f}")  
    print(f"Gamma Standard: {se_gamma:.6f}")
    print(f"ADF Test Statistic : {adf_stat:.6f}") #Lower than -2.86 is stationary, greater than -2.86 non-stationary
    if adf_stat > -2.86:
        print(f"{ticker_name} is non-stationary")
    else:
        print(f"{ticker_name} is stationary")
    print("\n")

--- Results ---
Results for GARAN: 
Gamma (γ): -0.000924
Gamma Standard: 0.001173
ADF Test Statistic : -0.787757
GARAN is non-stationary


Results for AKBNK: 
Gamma (γ): -0.001378
Gamma Standard: 0.001403
ADF Test Statistic : -0.982356
AKBNK is non-stationary


Results for ISCTR: 
Gamma (γ): -0.001828
Gamma Standard: 0.001384
ADF Test Statistic : -1.320594
ISCTR is non-stationary


Results for YKBNK: 
Gamma (γ): -0.001686
Gamma Standard: 0.001481
ADF Test Statistic : -1.138280
YKBNK is non-stationary


Results for TUPRS: 
Gamma (γ): 0.002000
Gamma Standard: 0.001151
ADF Test Statistic : 1.737600
TUPRS is non-stationary


Results for EREGL: 
Gamma (γ): -0.000977
Gamma Standard: 0.002188
ADF Test Statistic : -0.446229
EREGL is non-stationary


Results for KCHOL: 
Gamma (γ): -0.001522
Gamma Standard: 0.001330
ADF Test Statistic : -1.144533
KCHOL is non-stationary


Results for SAHOL: 
Gamma (γ): -0.001970
Gamma Standard: 0.001429
ADF Test Statistic : -1.378247
SAHOL is non-stationary


Re

Estimating the long run Relationship using pairs

In [93]:
df = pd.read_csv("../data/paired_stocks.csv")
for i in range(0, len(df)):
    stock1_name = df["Stock1"][i].replace(".IS", "")
    stock2_name = df["Stock2"][i].replace(".IS", "")

    df_1 = pd.read_csv((f"../data/{stock1_name}.csv"))
    df_2 = pd.read_csv((f"../data/{stock2_name}.csv"))

    merged_df = pd.merge(df_1[["Price", "Close"]], df_2[["Price", "Close"]], on="Price", suffixes=("_1", "_2")).dropna()

    Y_t = merged_df['Close_1'].values
    ones_col = np.ones(len(merged_df))

    M = np.column_stack((ones_col, merged_df['Close_2'].values))
    M_t = M.T

    M_multiplication = M_t @ M
    inv_M_multiplication = np.linalg.inv(M_multiplication)

    beta = inv_M_multiplication @ M_t @ Y_t

    e_t = Y_t - (beta[0] + beta[1] * X_t)

    e_t_array = e_t.values if hasattr(e_t, "values") else np.array(e_t)
    Ye_t = e_t_array[1:] - e_t_array[:-1]
    Xe_t = e_t_array[:-1]

    gamma = (Xe_t @ Ye_t) / (Xe_t @ Xe_t)

    Y_predicted = gamma * Xe_t
    u = Ye_t - Y_predicted

    N = len(Ye_t)

    sigma_squared = (u @ u) / (N - 1)
    SE_gamma = np.sqrt(sigma_squared / (Xe_t @ Xe_t)) # Standard Error gamma

    t_stat = gamma / SE_gamma

    print(f"t-stat: {t_stat:.4f}")

    #MacKinnon Values
    CRITICAL_1_PERCENT = -3.39
    CRITICAL_5_PERCENT = -2.76
    CRITICAL_10_PERCENT = -2.45

    if t_stat < CRITICAL_1_PERCENT:
        print(f"{stock1_name} and {stock2_name} is strongly cointegrated, strong long-term relationship")
    elif t_stat < CRITICAL_5_PERCENT:
        print(f"{stock1_name} and {stock2_name} is moderately cointegrated, significant long-term relationship")
    elif t_stat < CRITICAL_10_PERCENT:
        print(f"{stock1_name} and {stock2_name} is weakly cointegrated, relationship is fragile")
    else:
        print(f"{stock1_name} and {stock2_name} is not cointegrated, no long-term relationship")


t-stat: -2.7248
GARAN and AKBNK is weakly cointegrated, relationship is fragile
t-stat: 0.3725
ISCTR and YKBNK is not cointegrated, no long-term relationship
t-stat: 0.0493
GARAN and YKBNK is not cointegrated, no long-term relationship
t-stat: -0.5000
KCHOL and SAHOL is not cointegrated, no long-term relationship
t-stat: 0.2419
KCHOL and TUPRS is not cointegrated, no long-term relationship
t-stat: -2.9828
SAHOL and AKBNK is moderately cointegrated, significant long-term relationship
t-stat: -1.6353
EREGL and SISE is not cointegrated, no long-term relationship
t-stat: -0.0657
THYAO and TUPRS is not cointegrated, no long-term relationship
t-stat: -2.8079
BIMAS and SISE is moderately cointegrated, significant long-term relationship
